# PennyLane Hamiltonian simulation

Trotterize transverse-field Ising dynamics and compare an expectation-value trajectory.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    total_variation_distance,
)

In [ ]:
times = np.linspace(0.0, 1.2, 13)

def make_qnode(device):
    @qml.qnode(device)
    def evolution(time_value):
        qml.PauliX(0)
        steps = 6
        dt = time_value / steps
        for _ in range(steps):
            qml.IsingZZ(1.1 * dt, wires=[0, 1])
            qml.IsingZZ(1.1 * dt, wires=[1, 2])
            for wire in range(3):
                qml.RX(0.7 * dt, wires=wire)
        return qml.expval(qml.Z(0))
    return evolution

reference_qnode = make_qnode(qml.device("default.qubit", wires=3))
reference, reference_ms, _ = benchmark(lambda: np.asarray([reference_qnode(value) for value in times]))
mettleq_device = MettleQDevice(wires=3, method="statevector", device="cpu")
mettleq_qnode = make_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: np.asarray([mettleq_qnode(value) for value in times]))
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)
tutorial_result = emit_result(
    notebook="pennylane/11_hamiltonian_simulation.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="observable trajectory atol=3e-6",
    passed=error <= 3e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_observable_error": error, "times": times, "reference": reference, "mettleq": candidate},
)